In [58]:
import firebase_admin
from firebase_admin import credentials, db
from datetime import datetime
from zoneinfo import ZoneInfo
import pandas as pd
import numpy as np

# Path ke file credential JSON Anda
cred_path = "D:/staklimjerukagung-firebase-adminsdk-kcfma-e091165a9b.json"
database_url = 'https://staklimjerukagung-default-rtdb.asia-southeast1.firebasedatabase.app/'

# 1. Inisialisasi Aplikasi Source (Default)
if not firebase_admin._apps:
    source_cred = credentials.Certificate(cred_path)
    firebase_admin.initialize_app(source_cred, {'databaseURL': database_url})
else:
    firebase_admin.get_app()

# 2. Inisialisasi Aplikasi Destination
try:
    dest_app = firebase_admin.get_app('destination')
except ValueError:
    dest_cred = credentials.Certificate(cred_path)
    dest_app = firebase_admin.initialize_app(dest_cred, {'databaseURL': database_url}, name='destination')

print("✅ Firebase Source dan Destination berhasil diinisialisasi.")

✅ Firebase Source dan Destination berhasil diinisialisasi.


In [59]:
# Tentukan rentang waktu
start_readable_date = "30-04-2026 08:54:00"
end_readable_date = "30-04-2026 10:48:00"
jakarta_tz = ZoneInfo("Asia/Jakarta")

# Konversi ke Unix Timestamp
start_timestamp = int(datetime.strptime(start_readable_date, "%d-%m-%Y %H:%M:%S").replace(tzinfo=jakarta_tz).timestamp())
end_timestamp = int(datetime.strptime(end_readable_date, "%d-%m-%Y %H:%M:%S").replace(tzinfo=jakarta_tz).timestamp())

# Ambil data dari source
source_ref = db.reference('/auto_weather_stat/id-03/data')
source_data = source_ref.get()

if source_data:
    print(f"✅ Berhasil mengunduh {len(source_data)} baris data dari sumber.")
else:
    print("⚠️ Tidak ada data di database sumber.")

✅ Berhasil mengunduh 1326998 baris data dari sumber.


In [75]:
# 1. Masukkan ke DataFrame
df = pd.DataFrame.from_dict(source_data, orient='index')

# 2. Ubah index menjadi integer (karena index-nya adalah timestamp)
df.index = df.index.astype(int)

# 3. Potong/Filter berdasarkan rentang waktu yang kita tentukan di Cell 2
df_filtered = df[(df.index >= start_timestamp) & (df.index <= end_timestamp)].copy()

# 4. Buang kolom yang tidak perlu (drop sensor mati/duplikat)
cols_to_drop = ['humi', 'temp', 'pres']
df_filtered = df_filtered.drop(columns=[c for c in cols_to_drop if c in df_filtered.columns])

# 5. Pastikan semua kolom bertipe numerik agar bisa dihitung
numeric_cols = ['dew', 'humidity', 'pressure', 'temperature', 'volt', 'rainfall', 'rainrate']
for col in numeric_cols:
    if col in df_filtered.columns:
        df_filtered[col] = pd.to_numeric(df_filtered[col], errors='coerce')

print(f"✅ Sisa data setelah filter waktu dan kolom: {len(df_filtered)} baris.")
df_filtered.head() # Menampilkan sampel data sebelum kalibrasi

✅ Sisa data setelah filter waktu dan kolom: 113 baris.


,dew,humidity,pressure,temperature,timestamp,volt,rainfall,rainrate
1777514095,27.19990,95.04,1008.84,28.07,1777514095,4.19,0.0,0.0
1777514155,27.15688,94.69,1008.84,28.09,1777514155,4.18,0.0,0.0
1777514215,27.17138,94.55,1008.83,28.13,1777514215,4.19,0.0,0.0
1777514275,27.15784,94.31,1008.86,28.16,1777514275,4.18,0.0,0.0
1777514335,26.98540,93.36,1009.04,28.16,1777514335,4.18,0.0,0.0


In [76]:
# Tentukan nilai kalibrasi (offset)
calibration_offsets = {
    "pressure": 2,      # Ditambah 1.0
    "temperature": 0,  # Dikurangi 0.3
    "humidity": 0.0,      
    "dew": 0.0            
}

print("Menerapkan rumus kalibrasi...")

# Eksekusi kalibrasi ke kolom yang sesuai
for col, offset in calibration_offsets.items():
    if col in df_filtered.columns:
        # Tambahkan offset dan bulatkan 2 angka di belakang koma agar rapi
        df_filtered[col] = (df_filtered[col] + offset).round(2)

print("✅ Kalibrasi selesai. Berikut sampel hasilnya:")
df_filtered.head()

Menerapkan rumus kalibrasi...
✅ Kalibrasi selesai. Berikut sampel hasilnya:


,dew,humidity,pressure,temperature,timestamp,volt,rainfall,rainrate
1777514095,27.20,95.04,1010.84,28.07,1777514095,4.19,0.0,0.0
1777514155,27.16,94.69,1010.84,28.09,1777514155,4.18,0.0,0.0
1777514215,27.17,94.55,1010.83,28.13,1777514215,4.19,0.0,0.0
1777514275,27.16,94.31,1010.86,28.16,1777514275,4.18,0.0,0.0
1777514335,26.99,93.36,1011.04,28.16,1777514335,4.18,0.0,0.0


In [77]:
# 1. Ubah kembali DataFrame ke format dictionary
raw_dict = df_filtered.to_dict(orient='index')
final_data = {}

# 2. Bersihkan nilai NaN (Not a Number)
# Firebase akan error jika dikirim nilai NaN dari Pandas. Kita harus menghapusnya.
for ts, row in raw_dict.items():
    clean_row = {k: v for k, v in row.items() if pd.notna(v)}
    # Kunci dictionary di Firebase harus string
    final_data[str(ts)] = clean_row 

# 3. Referensi ke lokasi tujuan
dest_ref = db.reference('/auto_weather_stat/id-05/data', app=dest_app)

# 4. Eksekusi pengiriman data
if final_data:
    dest_ref.update(final_data)
    print(f"🎉 SUKSES! {len(final_data)} baris data yang sudah dikalibrasi berhasil dipindahkan ke id-05.")
else:
    print("⚠️ Tidak ada data yang memenuhi kriteria untuk dikirim.")

🎉 SUKSES! 113 baris data yang sudah dikalibrasi berhasil dipindahkan ke id-05.
